# Sprint 4 Campus — Connect FieldCare to a companion UI

Use this notebook alongside C20–C23. Build and inspect the screen in Lovable; use this notebook to observe the actual FieldCare API contract and a real stream. A notebook API call is a separate caller. It does **not** prove that the Lovable screen or its Edge Function is connected; verify that path in the Lovable preview and match the UI request ID to the approved safe service event.

The service remains the Module A application path: ChromaDB retrieves the service evidence, LangGraph coordinates retrieval and generation, and OpenRouter supplies the actual model response. A deterministic clarification does not call the model.

**Quota:** the supported example makes one real model-backed request. Keep the synthetic question and avoid rerunning it unless needed.

## Campus lessons in Notion

- [Build a focused FieldCare companion UI](https://app.notion.com/p/3ee5a55972ad8127a17fcbe5c5c6a2f2)
- [Connect the companion UI to your FieldCare service](https://app.notion.com/p/3ee5a55972ad81bd9ff7ce340aa02521)
- [Test the FieldCare integration from screen to service](https://app.notion.com/p/3ee5a55972ad8147958ddf184e9b31f8)
- [Document FieldCare for its next caller](https://app.notion.com/p/3ee5a55972ad814e838cc13ba3894ca6)

## 1. Set up the provisioned service connection

The course team supplies one reachable HTTPS service origin and a caller key. Store them in Colab Secrets as `FIELDCARE_SERVICE_URL` and `FIELDCARE_CALLER_KEY`. The notebook is a caller, so it does not need `OPENROUTER_API_KEY`; that provider credential stays with FieldCare.

In [ ]:
# One-time setup: use the course-provisioned reachable Sprint 3 service.
import json
import time
from urllib.parse import urlsplit

import requests
from google.colab import userdata

SERVICE_URL = (userdata.get("FIELDCARE_SERVICE_URL") or "").strip().rstrip("/")
CALLER_KEY = (userdata.get("FIELDCARE_CALLER_KEY") or "").strip()

parts = urlsplit(SERVICE_URL)
if parts.scheme != "https" or not parts.netloc or parts.path not in ("", "/"):
    raise RuntimeError("Add the course HTTPS service origin to Colab Secrets as FIELDCARE_SERVICE_URL.")
if not CALLER_KEY:
    raise RuntimeError("Add the course caller credential to Colab Secrets as FIELDCARE_CALLER_KEY.")

print("Course service settings loaded; credential values are hidden.")


## 2. Check process readiness

`/health` confirms that the configured service process is answering. It does not prove that the UI connector works or guarantee that a later model request will succeed.

In [ ]:
health = requests.get(f"{SERVICE_URL}/health", timeout=(5, 10))
print("GET /health:", health.status_code, health.json())

## 3. Follow an actual streamed answer

Before running the cell, predict which request fields the form must send. This supported synthetic request passes the FieldCare contract and reaches the real retrieval and generation path. Watch the `metadata`, actual provider text, and terminal event arrive. A partial answer without `complete` is not a completed result.

In [ ]:
def call_fieldcare(payload, *, authenticate=True, label="request"):
    """Call the real Sprint 3 API; print only the response produced by this run."""
    headers = {"Accept": "application/x-ndjson, application/json"}
    if authenticate:
        headers["X-API-Key"] = CALLER_KEY
    record = {
        "label": label,
        "http_status": None,
        "request_id": None,
        "content_type": None,
        "application_status": None,
        "citations": [],
        "terminal_event": None,
        "provider_delta_count": 0,
        "answer": "",
    }
    try:
        with requests.post(
            f"{SERVICE_URL}/v1/diagnose-stream",
            json=payload,
            headers=headers,
            stream=True,
            timeout=(5, 45),
        ) as response:
            record["http_status"] = response.status_code
            record["request_id"] = response.headers.get("X-Request-ID")
            record["content_type"] = response.headers.get("Content-Type", "")
            if response.status_code != 200:
                try:
                    record["error_detail"] = response.json().get("detail", "")
                except (ValueError, AttributeError):
                    record["error_detail"] = "Response body was not JSON."
                print(label, "HTTP", record["http_status"], record["error_detail"])
                return record

            if "application/x-ndjson" not in record["content_type"]:
                body = response.json()
                record["application_status"] = body.get("status")
                record["citations"] = body.get("citations", [])
                record["answer"] = body.get("answer", "")
                print(label, "HTTP", response.status_code, "application status:", record["application_status"])
                print("Actual JSON response:", body)
                return record

            deadline = time.monotonic() + 90
            for line in response.iter_lines(decode_unicode=True):
                if time.monotonic() > deadline:
                    raise TimeoutError("No terminal stream event arrived within 90 seconds.")
                if not line:
                    continue
                event = json.loads(line)
                event_type = event.get("type")
                if event_type == "metadata":
                    record["request_id"] = event.get("request_id", record["request_id"])
                    record["application_status"] = event.get("status")
                    record["citations"] = event.get("citations", [])
                    print("metadata:", {key: event.get(key) for key in ("status", "citations", "mode", "request_id")})
                elif event_type == "delta":
                    piece = event.get("text", "")
                    record["answer"] += piece
                    record["provider_delta_count"] += 1
                    print(piece, end="", flush=True)
                elif event_type == "usage":
                    print("\nusage:", {key: event.get(key) for key in ("model", "tokens")})
                elif event_type in ("complete", "error"):
                    record["terminal_event"] = event_type
                    print("\nterminal event:", event)
            if record["terminal_event"] is None:
                print("\nThe connection ended without a terminal event; treat this as incomplete.")
    except (requests.RequestException, TimeoutError, ValueError) as error:
        record["failure"] = type(error).__name__
        print(label, "request did not complete:", type(error).__name__)
    return record

supported = {
    "question": "Which filter and airflow checks are documented?",
    "equipment_id": "EQ-FC-1002",
}
supported_observation = call_fieldcare(supported, label="supported question")


### Read the event sequence

Use the values from your run. The `metadata` event supplies the status, citation IDs, mode, and request ID. `delta` events carry the text received from OpenRouter. `complete` marks a successful stream; `error` or a missing terminal event means the answer is incomplete. The wording and number of deltas vary between runs.

In [ ]:
print({
    "HTTP status": supported_observation["http_status"],
    "application status": supported_observation["application_status"],
    "request ID": supported_observation["request_id"],
    "citations": supported_observation["citations"],
    "terminal event": supported_observation["terminal_event"],
    "provider delta count": supported_observation["provider_delta_count"],
})

## 4. Compare a valid clarification

This request omits optional equipment context. It should pass input validation and reach FieldCare’s deterministic clarification branch. The JSON answer is service-authored clarification text, not a model response. Compare its HTTP and application statuses with the streamed request; both can use HTTP `200` while taking different application paths.

In [ ]:
incomplete = {"question": "Which airflow checks should the technician review?"}
clarification_observation = call_fieldcare(incomplete, label="valid request missing context")

## 5. Verify the API boundary

A missing required `question` should be rejected by FastAPI before the route or model path. This authenticated invalid attempt may count against the configured request allowance, so run it only if the course test plan includes it.

In [ ]:
invalid_observation = call_fieldcare({}, label="schema-invalid request")

## 6. Carry evidence into the UI check

Keep the direct Colab request and Lovable request separate. In Lovable, submit the same synthetic question through `fieldcareClient`; record the request ID shown by that response and find its matching sanitized service event using the course event view. Do not copy the notebook request ID as UI evidence.

In the README, document the exact route, request fields, stream event order, clarification JSON behavior, and the real observations from your own UI run. Do not paste credentials or customer text into the evidence record.

In [ ]:
from datetime import datetime, timezone
from google.colab import files

campus_evidence = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_is_direct_api_caller": True,
    "supported": {key: supported_observation.get(key) for key in (
        "http_status", "request_id", "content_type", "application_status",
        "citations", "terminal_event", "provider_delta_count", "failure"
    )},
    "clarification": {key: clarification_observation.get(key) for key in (
        "http_status", "request_id", "content_type", "application_status", "citations", "failure"
    )},
    "invalid": {key: invalid_observation.get(key) for key in (
        "http_status", "request_id", "content_type", "error_detail", "failure"
    )},
    "note": "This export records notebook-to-service observations; it is not evidence of the Lovable UI path."
}
evidence_path = "/content/sprint-4-campus-api-evidence.json"
with open(evidence_path, "w", encoding="utf-8") as stream:
    json.dump(campus_evidence, stream, indent=2)
files.download(evidence_path)